In [1]:
# Install Pytorch & other libraries
%pip install "torch==2.4.1" tensorboard 
%pip install flash-attn "setuptools<71.0.0" scikit-learn 
 
# Install Hugging Face libraries
%pip install  --upgrade \
  "datasets==3.1.0" \
  "accelerate==1.2.1" \
  "hf-transfer==0.1.8"
  #"transformers==4.47.1" \
 
# ModernBERT is not yet available in an official release, so we need to install it from github
%pip install "git+https://github.com/huggingface/transformers.git@6e0515e99c39444caae39472ee1b2fd76ece32f1" --upgrade

  Obtaining dependency information for torch==2.4.1 from https://files.pythonhosted.org/packages/cc/df/5204a13a7a973c23c7ade615bafb1a3112b5d0ec258d8390f078fa4ab0f7/torch-2.4.1-cp312-cp312-manylinux1_x86_64.whl.metadata
  Obtaining dependency information for tensorboard from https://files.pythonhosted.org/packages/5d/12/4f70e8e2ba0dbe72ea978429d8530b0333f0ed2140cc571a48802878ef99/tensorboard-2.19.0-py3-none-any.whl.metadata
  Obtaining dependency information for sympy from https://files.pythonhosted.org/packages/99/ff/c87e0622b1dadea79d2fb0b25ade9ed98954c9033722eb707053d310d4f3/sympy-1.13.3-py3-none-any.whl.metadata
  Obtaining dependency information for networkx from https://files.pythonhosted.org/packages/b9/54/dd730b32ea14ea797530a4479b2ed46a6fb250f682a9cfb997e968bf0261/networkx-3.4.2-py3-none-any.whl.metadata
  Obtaining dependency information for nvidia-cuda-nvrtc-cu12==12.1.105 from https://files.pythonhosted.org/packages/b6/9f/c64c03f49d6fbc56196664d05dba14e3a561038a81a638eeb47f4

In [1]:
from datasets import load_dataset
 
# Dataset id from huggingface.co/dataset
dataset_id = "youralien/feedback_qesconv_16wayclassification"
 
# Load raw dataset
raw_dataset = load_dataset(dataset_id, split="train") # happens to be called train

print(f"Raw dataset size: {len(raw_dataset)}")

/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8179/8179 [00:00<00:00, 14844.19 examples/s]


Raw dataset size: 8179


In [2]:
split_dataset = raw_dataset.train_test_split(test_size=0.1)
print(f"Train dataset size: {len(split_dataset['train'])}")
print(f"Test dataset size: {len(split_dataset['test'])}")
split_dataset['train'][0]

Train dataset size: 7361
Test dataset size: 818


{'conv_index': 194,
 'helper_index': 8,
 'input': ["Helper: I see. That's a very important role for any business. I bet there are numerous other retailers out there looking for a good manager to hire.",
  'Seeker: That is true, I am just not sure if that is what I want to do anymore',
  'Helper: I see. Well then, if this helps. The store closing might have been a positive thing, so you could find your next passion.',
  'Seeker: That is a good point.',
  'Helper: I will also add, that being a Manager means you have some very important skills that would translate well to your next job, or anything you decide to do.',
  "Seeker: That's true. I know the experience will help me but I still miss it",
  "Helper: Also, let's say you decided to work for another company, something new, outside of retail. Because you were a manager, you could also be promoted more quickly because of your management experience.",
  'Seeker: That is also true. I guess I just need to decide what I want to do now. Th

In [170]:
def prepare_input_text(example):
    # Convert the last two items of input list to a single text
    return {
        'text': "\n".join(example['input'][-2:]),
        **{k:v for k,v in example.items() if k != 'input'}  # Keep other fields
    }

# Apply the preprocessing
split_dataset = split_dataset.map(prepare_input_text)
split_dataset['train'][0]

Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 818/818 [00:00<00:00, 5429.85 examples/s]


{'conv_index': 216,
 'helper_index': 1,
 'input': ['Helper: hi! Hope you are doing well today. How may I assist you ?',
  'Seeker: Hey! I’ve been better. Just so stressed out! I don’t handle pressure well.',
  "Helper: I'm sorry to hear you're feeling stressed. Can you tell me more about what's causing this pressure?"],
 'Reflections-goodareas': 1,
 'Validation-goodareas': 0,
 'Empathy-goodareas': 0,
 'labels': 1,
 'Suggestions-goodareas': 0,
 'Self-disclosure-goodareas': 0,
 'Structure-goodareas': 0,
 'Professionalism-goodareas': 0,
 'Reflections-badareas': 0,
 'Validation-badareas': 0,
 'Empathy-badareas': 0,
 'Questions-badareas': 0,
 'Suggestions-badareas': 0,
 'Self-disclosure-badareas': 0,
 'Structure-badareas': 0,
 'Professionalism-badareas': 0,
 'text': "Seeker: Hey! I’ve been better. Just so stressed out! I don’t handle pressure well.\nHelper: I'm sorry to hear you're feeling stressed. Can you tell me more about what's causing this pressure?"}

In [171]:
from transformers import AutoTokenizer
 
# Model id to load the tokenizer
model_id = "answerdotai/ModernBERT-large"
# model_id = "answerdotai/ModernBERT-base"
# model_id = "google-bert/bert-base-uncased"

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.model_max_length = 512 # set model_max_length to 512 as prompts are not longer than 1024 tokens
 
# Tokenize helper 
 
# Tokenize helper function
def tokenize(batch):
    # return tokenizer(batch['text'], padding=True, truncation=True, return_tensors="pt")
    return tokenizer(batch['text'], padding='max_length', truncation=True, return_tensors="pt")


which_class = "Reflections-goodareas"
SKILL_OPTIONS = ["Reflections", "Validation", "Empathy", "Questions", "Suggestions", "Self-disclosure", "Structure", "Professionalism"]
goodareas_to_ignore = [f"{skill}-goodareas" for skill in SKILL_OPTIONS if f"{skill}-goodareas" != which_class]
badareas_to_ignore = [f"{skill}-badareas" for skill in SKILL_OPTIONS if f"{skill}-badareas" != which_class]
cols_to_remove = ['conv_index', 'helper_index', 'input', 'text']
cols_to_remove.extend(goodareas_to_ignore)
cols_to_remove.extend(badareas_to_ignore)
if which_class in split_dataset["train"].features.keys():
    split_dataset =  split_dataset.rename_column(which_class, "labels") # to match Trainer
tokenized_dataset = split_dataset.map(tokenize, batched=True, remove_columns=cols_to_remove)
 
tokenized_dataset["train"].features.keys()
# dict_keys(['labels', 'input_ids', 'attention_mask'])

ValueError: New column name labels already in the dataset. Please choose a column name which is not already in the dataset. Current columns in the dataset: ['conv_index', 'helper_index', 'input', 'Reflections-goodareas', 'Validation-goodareas', 'Empathy-goodareas', 'labels', 'Suggestions-goodareas', 'Self-disclosure-goodareas', 'Structure-goodareas', 'Professionalism-goodareas', 'Reflections-badareas', 'Validation-badareas', 'Empathy-badareas', 'Questions-badareas', 'Suggestions-badareas', 'Self-disclosure-badareas', 'Structure-badareas', 'Professionalism-badareas', 'text']

In [148]:
from transformers import AutoModelForSequenceClassification
 
# Prepare model labels - useful for inference
labels = ["not selected", "selected"]
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label
 
# Download the model from huggingface.co/models
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=num_labels, label2id=label2id, id2label=id2label,
)

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [149]:
import evaluate
import numpy as np

def compute_metrics_fn(eval_preds):
    metrics = dict()
    
    accuracy_metric = evaluate.load('accuracy')
    precision_metric = evaluate.load('precision')
    recall_metric = evaluate.load('recall')
    f1_metric = evaluate.load('f1')
    
    logits = eval_preds.predictions
    labels = eval_preds.label_ids
    preds = np.argmax(logits, axis=-1)  
    
    metrics.update(accuracy_metric.compute(predictions=preds, references=labels))
    metrics.update(precision_metric.compute(predictions=preds, references=labels, average='binary'))
    metrics.update(recall_metric.compute(predictions=preds, references=labels, average='binary'))
    metrics.update(f1_metric.compute(predictions=preds, references=labels, average='binary'))
    
    return metrics


In [150]:
class_distribution = split_dataset['train'].select_columns(['labels']).to_pandas().value_counts()
print("Class distribution:")
class_distribution = class_distribution / len(split_dataset['train'])
print(class_distribution)
inverse_weights = 1 / class_distribution
inverse_weights = inverse_weights.astype('float32')
inverse_weights.values

# https://github.com/richardaecn/class-balanced-loss/blob/master/src/cifar_main.py#L425-L430
# https://openaccess.thecvf.com/content_CVPR_2019/papers/Cui_Class-Balanced_Loss_Based_on_Effective_Number_of_Samples_CVPR_2019_paper.pdf#page=4.43
# hyperparameter sweep: {softmax, sigmoid, focal} for loss type, β ∈ {0.9, 0.99, 0.999, 0.9999} (Section 4), and γ ∈ {0.5, 1.0, 2.0} for focal loss [27].
# img_num_per_cls = data_utils.get_img_num_per_cls(
#     hparams['data_version'], hparams['imb_factor'])
# effective_num = 1.0 - np.power(hparams['beta'], img_num_per_cls)
# weights = (1.0 - hparams['beta']) / np.array(effective_num)
# weights = weights / np.sum(weights) * int(hparams['data_version'])


Class distribution:
labels
0         0.757642
1         0.242358
Name: count, dtype: float64


array([1.3198853, 4.126121 ], dtype=float32)

In [152]:
from huggingface_hub import HfFolder
from transformers import Trainer, TrainingArguments

import torch

def compute_class_balanced_loss(outputs, labels, num_items_in_batch):
    """depends on the class_distribution variable defined above"""
    logits = outputs['logits']
    criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor(inverse_weights.values, device=0))
    loss = criterion(logits, labels)
    return loss

# Define training args
training_args = TrainingArguments(
    output_dir= f"ModernBERT-{which_class}-classifier",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=7e-5, # originally 5e-5
    num_train_epochs=5, # suggestion by claude to reduce
    # weight_decay=0.01, # suggestion by claude to reduce
    bf16=True, # bfloat16 training 
    optim="adamw_torch_fused", # improved optimizer 
    # logging & evaluation strategies
    logging_strategy="steps",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    # use_mps_device=True, # mps device is a mac thing
    metric_for_best_model="f1",
    # push to hub parameters
    report_to="tensorboard",
    push_to_hub=True,
    hub_strategy="every_save",
    hub_token=HfFolder.get_token(),
)
 
 
# Create a Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics_fn,
    compute_loss_func=compute_class_balanced_loss, # custom class balanced loss
)

trainer.train()
# {'train_runtime': 3642.7783, 'train_samples_per_second': 1.235, 'train_steps_per_second': 0.04, 'train_loss': 0.535627057634551, 'epoch': 5.0}


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.444200,0.466718,0.745721,0.503916,0.914692,0.649832
2,0.377300,0.693698,0.808068,0.623853,0.644550,0.634033
3,0.283300,0.766353,0.805623,0.616071,0.654028,0.634483
4,0.171900,1.099500,0.783619,0.581731,0.573460,0.577566
5,0.091000,1.587234,0.782396,0.586387,0.530806,0.557214


TrainOutput(global_step=2305, training_loss=0.27855713936356813, metrics={'train_runtime': 598.1122, 'train_samples_per_second': 61.535, 'train_steps_per_second': 3.854, 'total_flos': 3.892334898514944e+16, 'train_loss': 0.27855713936356813, 'epoch': 5.0})

In [167]:
# Save processor and create model card
tokenizer.save_pretrained(f"ModernBERT-{which_class}-classifier")
trainer.create_model_card()
trainer.push_to_hub()

KeyboardInterrupt: 

## Exploring Hyperparameter Sweeps with WanDB
Link: https://wandb.ai/matt24/vit-snacks-sweeps/reports/Hyperparameter-Search-for-HuggingFace-Transformer-Models--VmlldzoyMTUxNTg0

In [13]:
# %pip install -qq wandb --upgrade
%pip install evaluate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Obtaining dependency information for evaluate from https://files.pythonhosted.org/packages/a2/e7/cbca9e2d2590eb9b5aa8f7ebabe1beb1498f9462d2ecede5c9fd9735faaf/evaluate-0.4.3-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 10.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [18]:
import wandb
wandb.login()

%env WANDB_PROJECT=ModernBert_SkillClassifier
%env WANDB_LOG_MODEL=checkpoint

env: WANDB_PROJECT=ModernBert_SkillClassifier
env: WANDB_LOG_MODEL=checkpoint


In [9]:
# method
sweep_config = {
    'method': 'random'
}


# hyperparameters
parameters_dict = {
    'epochs': {
        'value': 1
        },
    'batch_size': {
        'values': [8, 16, 32, 64]
        },
    'learning_rate': {
        'distribution': 'log_uniform_values',
        'min': 1e-5,
        'max': 1e-3
    },
    'weight_decay': {
        'values': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
    },
}


sweep_config['parameters'] = parameters_dict


In [20]:
sweep_id = wandb.sweep(sweep_config, project=f'modernbert-{which_class}-sweeps')

Create sweep with ID: np9rpy0r
Sweep URL: https://wandb.ai/ryanlouie2021-stanford-university/modernbert-Reflections-goodareas-sweeps/sweeps/np9rpy0r


In [15]:
from transformers import Trainer, TrainingArguments
import torch

def compute_class_balanced_loss(outputs, labels, num_items_in_batch):
    """depends on the class_distribution variable defined above"""
    logits = outputs['logits']
    criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor(inverse_weights.values, device=0))
    loss = criterion(logits, labels)
    return loss

def train(config=None):
    with wandb.init(config=config):
        # set sweep configuration
        config = wandb.config

        # Define training args
        training_args = TrainingArguments(
            output_dir= f"ModernBERT-{which_class}-classifier-sweeps",
            per_device_train_batch_size=config.batch_size,
            per_device_eval_batch_size=16,
            learning_rate=config.learning_rate,
            num_train_epochs=config.epochs,
            weight_decay=config.weight_decay,
            bf16=True, # bfloat16 training 
            optim="adamw_torch_fused", # improved optimizer 
            # logging & evaluation strategies
            logging_strategy="epoch",
            logging_steps=100,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=2,
            load_best_model_at_end=True,
            # use_mps_device=True, # mps device is a mac thing
            # push to hub parameters
            report_to="wandb",
            # push_to_hub=True,
            # hub_strategy="every_save",
            # hub_token=HfFolder.get_token(),
        )

        if config.use_class_balanced_loss:
            # Create a Trainer instance
            trainer = Trainer(
                model=model,
                args=training_args,
                train_dataset=tokenized_dataset["train"],
                eval_dataset=tokenized_dataset["test"],
                compute_metrics=compute_metrics_fn,
                compute_loss_func=compute_class_balanced_loss, # custom class balanced loss
            )
        else:
            trainer = Trainer(
                model=model,
                args=training_args,
                train_dataset=tokenized_dataset["train"],
                eval_dataset=tokenized_dataset["test"],
                compute_metrics=compute_metrics_fn,
            )

        trainer.train()

In [21]:
wandb.agent(sweep_id, train, count=20)

wandb: Agent Starting Run: 3u6o9mcs with config:
wandb: 	batch_size: 16
wandb: 	epochs: 1
wandb: 	learning_rate: 7.355714375771006e-05
wandb: 	weight_decay: 0.2
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: WARNING Config item 'learning_rate' was locked by 'sweep' (ignored update).
wandb: WARNING Config item 'weight_decay' was locked by 'sweep' (ignored update).
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Di

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.620900,0.478130,0.745721,0.901740,0.745721,0.792642


wandb: Adding directory to artifact (./ModernBERT-Reflections-goodareas-classifier-sweeps/checkpoint-461)... Done. 31.0s
wandb: Adding directory to artifact (./ModernBERT-Reflections-goodareas-classifier-sweeps/checkpoint-461)... Done. 37.8s
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


eval/accuracy,▁
eval/f1,▁
eval/loss,▁
eval/precision,▁
eval/recall,▁
eval/runtime,▁
eval/samples_per_second,▁
eval/steps_per_second,▁
train/epoch,▁▁▁
train/global_step,▁▁▁
train/grad_norm,▁


...

Run 0g9pz5x1 errored:
Traceback (most recent call last):
  File "/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/wandb/agents/pyagent.py", line 306, in _run_job
    self._function()
  File "/tmp/rylouie/ipykernel_2378267/119082754.py", line 51, in train
    trainer.train()
  File "/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/trainer.py", line 2163, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/trainer.py", line 2590, in _inner_training_loop
    self._maybe_log_save_evaluate(
  File "/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/trainer.py", line 3071, in _maybe_log_save_evaluate
    self.control = self.callback_handler.on_save(self.args, self.state, self.control)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/nlp/scr/rylouie/miniconda3

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
wandb: WARNING Config item 'learning_rate' was locked by 'sweep' (ignored update).
wandb: WARNING Config item 'weight_decay' was locked by 'sweep' (ignored update).


Epoch,Training Loss,Validation Loss


Traceback (most recent call last):
  File "/tmp/rylouie/ipykernel_2378267/119082754.py", line 51, in train
    trainer.train()
  File "/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/trainer.py", line 2163, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/trainer.py", line 2590, in _inner_training_loop
    self._maybe_log_save_evaluate(
  File "/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/trainer.py", line 3071, in _maybe_log_save_evaluate
    self.control = self.callback_handler.on_save(self.args, self.state, self.control)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/trainer_callback.py", line 508, in on_save
    return self.call_event("on_save", args, state, control)
          

Run noplgwdw errored:
Traceback (most recent call last):
  File "/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/wandb/agents/pyagent.py", line 306, in _run_job
    self._function()
  File "/tmp/rylouie/ipykernel_2378267/119082754.py", line 51, in train
    trainer.train()
  File "/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/trainer.py", line 2163, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/trainer.py", line 2590, in _inner_training_loop
    self._maybe_log_save_evaluate(
  File "/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/trainer.py", line 3071, in _maybe_log_save_evaluate
    self.control = self.callback_handler.on_save(self.args, self.state, self.control)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/nlp/scr/rylouie/miniconda3

## Using the model to make predictions

In [186]:
import pandas as pd


condition = "control"
# condition = "treatment"
# input_data = pd.read_csv(f"../Empathy-Mental-Health/dataset/all_{condition}_seekerhelper_pairs.csv")
input_data = pd.read_csv("N94_all_seekerhelper_pairs.csv")
print(len(input_data))
input_data.head()

3842


,id,seeker_post,response_post
0,14_a1df3c7155d0438b9c4084b57b66c6e6_0_0,NaN,good evening I understand you're feeling isola...
1,14_a1df3c7155d0438b9c4084b57b66c6e6_0_1,"Yeah, it's just... everyone was with their fam...",I hear you. It can be difficult growing apart ...
2,14_a1df3c7155d0438b9c4084b57b66c6e6_0_2,"Yeah, it's tough. You know, during the holiday...",Perhaps they're feeling similarly. Waiting for...
3,14_a1df3c7155d0438b9c4084b57b66c6e6_0_3,But why should I always have to be the bigger ...,"Being the bigger person can feel burdensome, c..."
4,14_a1df3c7155d0438b9c4084b57b66c6e6_0_4,"Yeah, exactly. It's like, why should I keep pu...",How would you like for them to show you they c...


In [187]:
from transformers import pipeline
 
# load model from huggingface.co/models using our repository id
classifier = pipeline("sentiment-analysis", model=f"ModernBERT-{which_class}-classifier", device=0)
# classifier = pipeline("sentiment-analysis", model="ModernBERT-Empathy-goodareas-classifier", device=0)

# sample = f"Seeker: {input_data.loc[0, "seeker_post"]}\nHelper: {input_data.loc[0, "response_post"]}"
# pred = classifier(sample)
# print(pred)

def binary_prediction_seeker_response_post(seeker, helper):
    sample = f"Seeker: {seeker}\nHelper: {helper}"
    pred = classifier(sample)
    return int(pred[0]['label'] == 'selected')

Device set to use cuda:0


In [188]:
# output_preds = input_data.apply(binary_prediction_seeker_response_post, axis=0)

strengths = [binary_prediction_seeker_response_post(input_data.loc[i, "seeker_post"], input_data.loc[i, "response_post"])
             for i in range(len(input_data))]

In [189]:
input_data[f"{which_class}"] = strengths
# input_data["Empathy-goodareas"] = strengths

In [193]:
print(len(input_data))
input_data.head()

3842


,id,seeker_post,response_post,Reflections-goodareas
0,14_a1df3c7155d0438b9c4084b57b66c6e6_0_0,NaN,good evening I understand you're feeling isola...,0
1,14_a1df3c7155d0438b9c4084b57b66c6e6_0_1,"Yeah, it's just... everyone was with their fam...",I hear you. It can be difficult growing apart ...,1
2,14_a1df3c7155d0438b9c4084b57b66c6e6_0_2,"Yeah, it's tough. You know, during the holiday...",Perhaps they're feeling similarly. Waiting for...,0
3,14_a1df3c7155d0438b9c4084b57b66c6e6_0_3,But why should I always have to be the bigger ...,"Being the bigger person can feel burdensome, c...",0
4,14_a1df3c7155d0438b9c4084b57b66c6e6_0_4,"Yeah, exactly. It's like, why should I keep pu...",How would you like for them to show you they c...,0


In [195]:
# input_data.to_csv(f'all_{condition}_seekerhelper_pairs_{which_class}.csv')
input_data.to_csv(f"N94_all_seekerhelper_pairs_{which_class}.csv")

In [192]:
f'all_{condition}_seekerhelper_pairs_{which_class}.csv'

'all_control_seekerhelper_pairs_Reflections-goodareas.csv'